# Running `llama.cpp` as a server with a quantized MoE model and multi-token-prediction

Now, we will try to increas the speed even more by using an MoE model which has
fewer active parameters. The model itself is larger though and needs to fit
inside the GPU. As `llama.cpp` handles GPU RAM so efficiently, we will give 
it a try.

We will use the OpenAI client:

In [ ]:
from openai import OpenAI

An API key can be added, but we don't need it here

In [ ]:
openai_api_key = "EMPTY"
openai_api_base = "http://localhost:8090/v1"
client = OpenAI(
    api_key=openai_api_key,
    base_url=openai_api_base,
)

Now the MoE model: `llama-server -hf unsloth/Qwen3.6-35B-A3B-MTP-GGUF:UD-Q4_K_XL -ngl 99 -c 8192 -fa on -np 1 --spec-type draft-mtp --spec-draft-n-max 3 --port 8090 --host 0.0.0.0`

As always with `llama.cpp` the startup is quite fast!

In [ ]:
model = "Qwen3.6-35B-A3B-MTP-GGUF:UD-Q4_K_XL"

In [ ]:
%%time
completion = client.chat.completions.create(
    model=model, 
    messages=[{ "role": "user",
                "content": "Explain O'Reilly online learning!" } ]
)

The model is very fast due to the "only" 3B active parameters!

In [ ]:
from IPython.display import display, Markdown
display(Markdown(completion.choices[0].message.content))

The `responses` API is also supported by `llama.cpp`. Let's generate `python` code, which supposedly can also
be created by the draft model:

In [ ]:
%%time
response = client.responses.create(model=model, 
                                   input="Write a quicksort in Python")

In [ ]:
display(Markdown(response.output_text))

The speed difference is not so significant here as the "big" model is much faster with the
3B parameters compared to the 27B in the previous model. Still, a solid gain.

Now, take a look at the GPU memory usage.

In [ ]:
!nvidia-smi

`llama.cpp` could even handle larger models due to its space efficiency.